# Step 1+2 (swaither = Bias Correction + Super-Resolution) Evaluation Notebook

This notebook is the **research artifact used to produce the final results
reported in the paper**. It loads generated diffusion samples (Step 1 + Step 2
combined), computes evaluation metrics on the validation set, and produces
the figures and tables shown in Section 4.

## What this notebook does

- Loads **generated diffusion samples** from `../outputs/generations/` for each
  Step 1 variant we compare in the paper (UNet λ=2, MultiFT, single-lead
  baselines, etc.).
- Loads **intermediate bias-corrected fields** from `../data/` (used to compute
  "before super-resolution" metrics and for visualization).
- Computes deterministic metrics (MSE, MSE-wet, CSI), probabilistic metrics
  (CRPS, FSS), spectral metrics (RAPSD, eS/eA/eL), PIT diagrams, and produces spider diagrams for metric comparison.

## Reproducing this notebook

This notebook expects:

1. **Generated diffusion samples** in `../outputs/generations/`. To produce them, run
   one of:
   - `scripts/inference_full_pipeline.py` — full pipeline starting from raw AIFS forecasts
     (AIFS → bias correction → super-resolution)
   - `scripts/generate_super_resolution.py` — Step 2 only, starting from
     pre-computed bias-corrected inputs

2. **Bias-corrected fields** in `../data/` (the
   `AIFS_bias_corrected_precip_VAL_*.nc` files). These are produced by running
   Step 1 inference on the validation set; included in the public release of
   the published model variants if you download them.

3. **Pretrained checkpoints** if you want to skip retraining. To use the checkpoints from the paper, contact the corresponding author for access (see README). Otherwise, retrain using the configs in ../conf/ and update the paths below to point to your local checkpoint files.

⚠️ **This notebook is provided for reproducibility and reference**, not as
a tutorial. The many generation outputs loaded below correspond to different
paper variants; for casual exploration of the final model, you only need the
λ=2 outputs (Cell 5, `root1`). However, it can be easily adapted to new trained models and new generated samples. To easily import generated samples from the full inference pipeline (until the visualization step), follow the "For standard NEW model samples from `inference_swAIther.py`" cells.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm

plt.style.use("ggplot")
plt.rcParams["lines.linewidth"] = 2

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from swaither.helpers.viz_helpers import (
    Viz_DiffSamplesMosaicMultiLead,
    Viz_AllLeadsOverview,
    PlotDistribValSet,
)
from swaither.helpers.post_processing_helpers import postprocess_with_lr_mask
from swaither.helpers.metrics_helpers import (
    compute_metrics_on_latlon_window_extended,
    compute_metrics_per_lead,
    compute_global_metrics,
    plot_metrics_by_lead,
    compute_pit_on_latlon_window,
    plot_pit_summary,
    plot_pit_leads_panel,
    plot_pit_comparison_multi,
    plot_kl_by_lead,
    plot_pit_by_intensity_multi,
    plot_pit_histogram_multi,
    plot_pit_comparison_combined,
)
from swaither.helpers.tables_helpers import (
    generate_ensemble_latex_table_compact,
    generate_ensemble_latex_table_per_lead,
    plot_metrics_by_lead_sr,
)
from swaither.helpers.viz_spider_plots import spider_plot_unified3

In [ ]:
DATA_ROOT = "../data"
GENERATIONS_ROOT = "../outputs/generations"
RESULTS_ROOT = "../outputs/images_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

## 1) Loading generated diffusion samples (Step 1 + 2)

We import the final diffusion samples from the super resolution model, in order to compare them with observed high resolution images. The format slightly changed, depending if the samples were generated from `scripts/inference_full_pipeline.py` or `scripts/generate_super_resolution.py`.

The cells below load diffusion samples for the different Step 1 variants
compared in the paper:

| Variable | Step 1 variant | Role in paper |
|---|---|---|
| `root1` / `truth1` / `pred1` | **UNet λ=2** | **Final SwAIther-Precip model** (Table X, headline results) |
| `root2` / `truth2` / `pred2` | UNet MultiFT | Multi-stage fine-tuned variant (Table X) |
| `root3` / `truth3` / `pred3` | UNet MultiFT, new inference | Alternative run for sanity check |
| `root_2023test` | UNet λ=2, yearly split | Generalization test on held-out 2023 (Section X) |
| `root_2023ValStride5` | UNet λ=2, yearly split | Validation subset on 2023 |
| `root_newHours` | UNet λ=2, 6h/18h test | Unobserved hours (Supplemental Material) |

For casual exploration, **only `root1` is needed** — it corresponds to the
final published model. Comment out the other loads if you don't have those
generation outputs.

Outputs from `generate_MultiHori_OneFile.py` (used on the paper)

In [ ]:
#######################################################
# Main test diffusion samples (weekly split, stride 4)
#######################################################

# UNet lambda=2
root1 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4/UNet_lam2_ValSampleStride4_samples.nc"
)
truth1, pred1 = root1["truth"], root1["pred"]

# UNet Multi-Step Fine-Tuning
root2 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples.nc"
)
truth2, pred2 = root2["truth"], root2["pred"]

# UNet MultiFT — new inference script
root_21 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples_newInfScript.nc"
)
pred_21 = root_21.pred
bias_corrected_21 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples_newInfScript_bias_corrected.nc"
)

# AIFS original predictions (no diffusion)
root3 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/AIFSOrigPreds_ValSampleStride4/AIFSOrigPreds_ValSampleStride4_samples.nc"
)
truth3, pred3 = root3["truth"], root3["pred"]

# Baseline B4: per-pixel multi-channel linear regression
root4 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/BaselineB4_ValSampleStride4/BaselineB4_ValSampleStride4_samples.nc"
)
truth4, pred4 = root4["truth"], root4["pred"]

# Single-lead models (6 h, 3 days, 6 days)
root5 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_6h_ValSampleStride4/SingleLead_6h_ValSampleStride4_samples.nc"
)
root6 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_3days_ValSampleStride4/SingleLead_3days_ValSampleStride4_samples.nc"
)
root7 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_6days_ValSampleStride4/SingleLead_6days_ValSampleStride4_samples.nc"
)
truth5, pred5 = root5["truth"], root5["pred"]
truth6, pred6 = root6["truth"], root6["pred"]
truth7, pred7 = root7["truth"], root7["pred"]
pred6 = pred6.sel(lead=[0])  # only first lead was trained for 3-day model

Outputs from `inference_swAIther.py` (used on the paper)

In [ ]:
#######################################################
# Yearly split diffusion samples
#######################################################

# Nov–Dec 2023 test
root_2023test = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/UNet_lam2_YearlySplit2_2023NovDec_samples.nc"
)
pred_2023test = root_2023test.pred
bias_corrected_2023test = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/UNet_lam2_YearlySplit2_2023NovDec_samples_bias_corrected.nc"
)

# 2023 validation stride-5
root_2023ValStride5 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/UNet_lam2_2023ValStride5_YearlySplitTest_samples.nc"
)
pred_2023ValStride5 = root_2023ValStride5.pred
bias_corrected_2023ValStride5 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/UNet_lam2_2023ValStride5_YearlySplitTest_samples_bias_corrected.nc"
)

For standard NEW model samples from `inference_swAIther.py`

In [ ]:
root_test = xr.open_dataset(
    f"{GENERATIONS_ROOT}/inference_test/samples.nc"
)
pred_test = root_test.pred
bias_corrected_test = xr.open_dataset(
    f"{GENERATIONS_ROOT}/inference_test/samples_bias_corrected.nc"
)

In [ ]:
#######################################################
# Extra test for the supplemental material: unobserved hours of the day (06 h / 18 h)
#######################################################

root_newHours = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/UNet_lam2_ValSampleStride4_6h_18hTest_samples.nc"
)
bias_corrected_newHours = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/UNet_lam2_ValSampleStride4_6h_18hTest_samples_bias_corrected.nc"
)

# Filter to times available in CombiPrecip (some days removed during preprocessing)
CombiPrecip_newHours = xr.open_dataset(
    f"{DATA_ROOT}/CPCH_processed_precip_2019_2023_concat_NotFlat_6hSUM_ValSampleStride4_6h_18hTest_wetEnough_gamma.nc"
)

t1 = CombiPrecip_newHours.time.values
t2 = pd.to_datetime(root_newHours.time.values)
common_times = np.intersect1d(t1, t2)
common_times_str = pd.DatetimeIndex(common_times).strftime("%Y-%m-%dT%H:%M:%S").values

root_newHours = root_newHours.sel(time=common_times_str)
pred_newHours = root_newHours.pred
bias_corrected_newHours = bias_corrected_newHours.sel(time=common_times)

### Import original low-res AIFS inputs

In [ ]:
AIFS_inputs = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc"
)

In [ ]:
# Weekly-split validation subset
AIFS_inputs_val = AIFS_inputs.sel(time=truth1.time)
AIFS_inputs_val_smallSample = AIFS_inputs.sel(time=truth1.time[:6])

# Yearly-split 2023 validation subsets
AIFS_inputs_2023 = AIFS_inputs.sel(time=AIFS_inputs.time[-685:])
AIFS_inputs_2023_val = AIFS_inputs_2023.sel(time=AIFS_inputs_2023.time[::5])
AIFS_inputs_2023_selection = AIFS_inputs_2023.sel(
    time=AIFS_inputs_2023.time[566 : 626 + 60]
)  # Nov–Dec

# AIFS inputs for the 06 h / 18 h extra test
AIFS_inputs_test_extra = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_inputs_ValSampleStride4_6h_18hTest.nc"
)
AIFS_inputs_new_hours = AIFS_inputs_test_extra.sel(time=pred_newHours.time)

In [ ]:
def pick_k_per_month_of_year(time_da, k=10, seed=0, replace_if_needed=False):
    """Sample up to k time steps per calendar month for balanced evaluation."""
    t = pd.DatetimeIndex(time_da.values)
    s = pd.Series(t, name="time")
    rng = np.random.default_rng(seed)
    out = []
    for m in range(1, 13):
        g = s[s.dt.month == m]
        if len(g) == 0:
            continue
        replace = replace_if_needed and (len(g) < k)
        chosen = rng.choice(
            g.to_numpy(), size=min(k, len(g)) if not replace else k, replace=replace
        )
        out.append(chosen)
    return np.sort(np.concatenate(out)).astype("datetime64[ns]")


selected = pick_k_per_month_of_year(AIFS_inputs_new_hours["time"], k=10, seed=42)
AIFS_inputs_new_hours_sub = AIFS_inputs_new_hours.sel(time=selected)
selected_str = pd.DatetimeIndex(selected).strftime("%Y-%m-%dT%H:%M:%S").values

Check temporal distribution of the validation set

In [ ]:
PlotDistribValSet(
    AIFS_inputs_val.time.values,
    img_name="timestep_distribution_ValSampleStride4_WeeklySplit.png",
)

### If not using the full inference pipeline: Load intermediate bias-corrected fields

The 7 files below are Step 1 outputs (bias-corrected, low-res, interpolated
onto the high-res grid). They're used to compute "before super-resolution"
metrics for each Step 1 variant.

| Variable | Step 1 variant |
|---|---|
| `bias_corrected_1` | **UNet λ=2 (final model)** |
| `bias_corrected_2` | UNet MultiFineTuning |
| `bias_corrected_3` | Raw AIFS (no bias correction; baseline) |
| `bias_corrected_4` | Baseline B4 (per-cell multivariate regression) |
| `bias_corrected_5` | UNet single-lead, 6h |
| `bias_corrected_6` | UNet single-lead, 3 days |
| `bias_corrected_7` | UNet single-lead, 6 days |

These files are produced by `swaither/helpers/bc_inference.py` functions, to be run within the `Step1_Eval_notebook_refactor.ipynb` or in a separated script, on the
validation set. For casual exploration, only `bias_corrected_1` is needed.

In [ ]:
bias_corrected_1 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_NoCurri_lam2_AllHori_interp_good.nc"
)
bias_corrected_2 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_MultiStepFineTuning_2_AllHori_interp_good.nc"
)
bias_corrected_3 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_AIFSOriginalPreds_AllHori_interp_good.nc"
)
bias_corrected_4 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_BaselineB4_AllHori_interp_good.nc"
)
bias_corrected_5 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_SingleLead_6h_interp_good.nc"
)
bias_corrected_6 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_SingleLead_3days_interp_good.nc"
)
bias_corrected_7 = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_SingleLead_6days_interp_good.nc"
)

### If using the full inference pipeline: import and put the CombiPrecip data in the right form. 

Import the CombiPrecip precipitation data (not included in outputs).

In [ ]:
CombiPrecip = xr.open_dataset(
    f"{DATA_ROOT}/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma_wElevation.nc"
)

Put the observed data in the right format the "truth" with the _crop_truth function

In [ ]:
# extract observevd data in the same window/format as the bias corrected data
def _crop_truth(combiprecip, bias_corrected_ref):
    """Select matching times, crop to lat/lon bounds, and standardise coordinates."""
    t = combiprecip.sel(time=bias_corrected_ref.time)
    t = t.sel(
        lat=slice(bias_corrected_ref.lat.values[0], bias_corrected_ref.lat.values[-1]),
        lon=slice(bias_corrected_ref.lon.values[0], bias_corrected_ref.lon.values[-1]),
    )
    return (
        t.precip6h_cumulative_capped.rename("truth")
        .rename({"lat": "y", "lon": "x"})
        .drop_vars(["y", "x"] + (["altitude"] if "altitude" in t.coords else []))
        .assign_coords(time=t.time.dt.strftime("%Y-%m-%dT%H:%M:%S"))
    )

In [ ]:
truth_2023test = _crop_truth(CombiPrecip, bias_corrected_2023test)
truth_2023ValStride5 = _crop_truth(CombiPrecip, bias_corrected_2023ValStride5)

In [ ]:
CombiPrecip_newHours = xr.open_dataset(
    f"{DATA_ROOT}/CPCH_processed_precip_2019_2023_concat_NotFlat_6hSUM_ValSampleStride4_6h_18hTest_wetEnough_gamma.nc"
)
truth_newHours = _crop_truth(CombiPrecip_newHours, bias_corrected_newHours)

# Final sub-selection for the new-hours test
pred_newHours_sel = pred_newHours.sel(time=selected_str)
truth_newHours_sel = truth_newHours.sel(time=selected_str)
bias_corrected_newHours_sel = bias_corrected_newHours.sel(time=selected)

For standard NEW model samples from `inference_swAIther.py`

In [ ]:
truth_test = _crop_truth(CombiPrecip, bias_corrected_test)

pred_test_sel = pred_test.sel(time=pred_test.time)
truth_test_sel = truth_test.sel(time=pred_test.time)
bias_corrected_test_sel = bias_corrected_test.sel(time=pred_test.time)

## 2) Post-process predictions and observations

Apply a low-resolution bias-predicted mask over generated samples and discard precipitation below 0.1 mm/h (measurement noise floor).

In [ ]:
# Weekly-split models
pred1_clean, truth1_clean = postprocess_with_lr_mask(
    pred=pred1, obs=truth1, lr_bias_corrected=bias_corrected_1, threshold=0.1
)
xr.Dataset({"truth": truth1_clean, "pred": pred1_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4/UNet_lam2_ValSampleStride4_samples_clean.nc"
)

pred2_clean, truth2_clean = postprocess_with_lr_mask(
    pred=pred2, obs=truth2, lr_bias_corrected=bias_corrected_2, threshold=0.1
)
xr.Dataset({"truth": truth2_clean, "pred": pred2_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples_clean.nc"
)

pred3_clean, truth3_clean = postprocess_with_lr_mask(
    pred=pred3, obs=truth3, lr_bias_corrected=bias_corrected_3, threshold=0.1
)
xr.Dataset({"truth": truth3_clean, "pred": pred3_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/AIFSOrigPreds_ValSampleStride4/AIFSOrigPreds_ValSampleStride4_samples_clean.nc"
)

In [ ]:
pred4_clean, truth4_clean = postprocess_with_lr_mask(
    pred=pred4, obs=truth4, lr_bias_corrected=bias_corrected_4, threshold=0.1
)
xr.Dataset({"truth": truth4_clean, "pred": pred4_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/BaselineB4_ValSampleStride4/BaselineB4_ValSampleStride4_samples_clean.nc"
)

pred5_clean, truth5_clean = postprocess_with_lr_mask(
    pred=pred5, obs=truth5, lr_bias_corrected=bias_corrected_5, threshold=0.1
)
xr.Dataset({"truth": truth5_clean, "pred": pred5_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/SingleLead_6h_ValSampleStride4/SingleLead_6h_ValSampleStride4_samples_clean.nc"
)

pred6_clean, truth6_clean = postprocess_with_lr_mask(
    pred=pred6, obs=truth6, lr_bias_corrected=bias_corrected_6, threshold=0.1
)
xr.Dataset({"truth": truth6_clean, "pred": pred6_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/SingleLead_3days_ValSampleStride4/SingleLead_3days_ValSampleStride4_samples_clean.nc"
)

pred7_clean, truth7_clean = postprocess_with_lr_mask(
    pred=pred7, obs=truth7, lr_bias_corrected=bias_corrected_7, threshold=0.1
)
xr.Dataset({"truth": truth7_clean, "pred": pred7_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/SingleLead_6days_ValSampleStride4/SingleLead_6days_ValSampleStride4_samples_clean.nc"
)

In [ ]:
# Yearly-split and extra tests
pred_2023test_clean, truth_2023test_clean = postprocess_with_lr_mask(
    pred=pred_2023test,
    obs=truth_2023test,
    lr_bias_corrected=bias_corrected_2023test,
    threshold=0.1,
)
xr.Dataset({"truth": truth_2023test_clean, "pred": pred_2023test_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/UNet_lam2_YearlySplit2_2023NovDec_samples_clean.nc"
)

pred_2023ValStride5_clean, truth_2023ValStride5_clean = postprocess_with_lr_mask(
    pred=pred_2023ValStride5,
    obs=truth_2023ValStride5,
    lr_bias_corrected=bias_corrected_2023ValStride5,
    threshold=0.1,
)
xr.Dataset(
    {"truth": truth_2023ValStride5_clean, "pred": pred_2023ValStride5_clean}
).to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/UNet_lam2_2023ValStride5_YearlySplitTest_samples_clean.nc"
)

pred21_clean, truth21_clean = postprocess_with_lr_mask(
    pred=pred_21, obs=truth2, lr_bias_corrected=bias_corrected_21, threshold=0.1
)
xr.Dataset({"truth": truth21_clean, "pred": pred21_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples_newInfScript_clean.nc"
)
pred_newHours_clean, truth_newHours_clean = postprocess_with_lr_mask(
    pred=pred_newHours_sel,
    obs=truth_newHours_sel,
    lr_bias_corrected=bias_corrected_newHours_sel,
    threshold=0.1,
)
xr.Dataset({"truth": truth_newHours_clean, "pred": pred_newHours_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/UNet_lam2_ValSampleStride4_6h_18hTest_samples_sel_clean.nc"
)

For standard NEW model samples from `inference_swAIther.py`

In [ ]:
# Weekly-split models
pred1_clean, truth1_clean = postprocess_with_lr_mask(
    pred=pred_test_sel, obs=truth_test_sel, lr_bias_corrected=bias_corrected_test_sel, threshold=0.1
)

xr.Dataset({"truth": truth1_clean, "pred": pred1_clean}).to_netcdf(
    f"{GENERATIONS_ROOT}/inference_test/samples_clean.nc"
)

## 3) Load back post-processed generated samples

In [ ]:
# Weekly-split models
root1 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4/UNet_lam2_ValSampleStride4_samples_clean.nc"
)
root2 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples_clean.nc"
)
root3 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/AIFSOrigPreds_ValSampleStride4/AIFSOrigPreds_ValSampleStride4_samples_clean.nc"
)
root4 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/BaselineB4_ValSampleStride4/BaselineB4_ValSampleStride4_samples_clean.nc"
)
root5 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_6h_ValSampleStride4/SingleLead_6h_ValSampleStride4_samples_clean.nc"
)
root6 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_3days_ValSampleStride4/SingleLead_3days_ValSampleStride4_samples_clean.nc"
)
root7 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_6days_ValSampleStride4/SingleLead_6days_ValSampleStride4_samples_clean.nc"
)

truth1, pred1 = root1["truth"], root1["pred"]
truth2, pred2 = root2["truth"], root2["pred"]
truth3, pred3 = root3["truth"], root3["pred"]
truth4, pred4 = root4["truth"], root4["pred"]
truth5, pred5 = root5["truth"], root5["pred"]
truth6, pred6 = root6["truth"], root6["pred"]
truth7, pred7 = root7["truth"], root7["pred"]
pred6 = pred6.sel(lead=[0])

# Yearly-split and extra tests
root_yearly = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/UNet_lam2_YearlySplit2_2023NovDec_samples_clean.nc"
)
truth_yearly, pred_yearly = root_yearly["truth"], root_yearly["pred"]

root_2023ValStride5 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/UNet_lam2_2023ValStride5_YearlySplitTest_samples_clean.nc"
)
truth_2023ValStride5, pred_2023ValStride5 = (
    root_2023ValStride5["truth"],
    root_2023ValStride5["pred"],
)

root21 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/UNet_MultiStepFineTuning2_ValSampleStride4_samples_newInfScript_clean.nc"
)
truth21, pred21 = root21["truth"], root21["pred"]

root_newHours = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/UNet_lam2_ValSampleStride4_6h_18hTest_samples_sel_clean.nc"
)
truth_newHours, pred_newHours = root_newHours["truth"], root_newHours["pred"]

For standard NEW model samples from `inference_swAIther.py`

In [ ]:
root_NEW = xr.open_dataset(
    f"{GENERATIONS_ROOT}/inference_test/samples_clean.nc"
)
truth21_NEW, pred_NEW = root_NEW["truth"], root_NEW["pred"]

## 4) Visualise generated downscaled samples

Median sample over all generated samples, for Multiple leads, given a valid time.

In [ ]:
fig = Viz_AllLeadsOverview(
    [pred1, pred2],
    pred_names=["UNet$_{\\lambda=2}$", "UNet Multi FT"],
    truth=truth1,
    lr=[bias_corrected_1, bias_corrected_2],
    AIFS_inputs=AIFS_inputs,
    index_time=2,
    pred_reduce="median",
)
# fig.savefig(f"{RESULTS_ROOT}/Diff_viz_AllLeads_comp.png", dpi=300, bbox_inches="tight")

Multiple generated samples and quantile samples, given one lead time and one valid time.

In [ ]:
fig = Viz_DiffSamplesMosaicMultiLead(
    [pred1, pred2],
    pred_names=["UNet$_{\\lambda=2}$", "UNet Multi FT"],
    truth=truth1,
    lr=[bias_corrected_1, bias_corrected_2],
    AIFS_inputs=AIFS_inputs,
    index_time=2,
    lead_index=4,
    sample_indices=[
        0,
        1,
        3,
        4,
    ],  # choose the samples to be shown out of the total number of generated samples (12 in the paper)
)
# fig.savefig(f"{RESULTS_ROOT}/Diff_samples_viz_indexTime_2_lead4.png", dpi=300, bbox_inches="tight")

Additional visualizations for the supplementary material: November event visualisation

In [ ]:
figs_nov = []
figs_nov_member = []

for i in range(23, 30):
    common_kwargs = dict(
        index_time=i,
        pred_var="precip6h_cumulative_capped",
        truth_var="precip6h_cumulative_capped",
        lr_var="precip6h_cumulative_capped",
        aifs_var="total_precipitation_NoNeg",
        max_leads=None,
        lead_indices=None,
        show_aifs=True,
        ncols=9,
    )
    figs_nov.append(
        Viz_AllLeadsOverview(
            pred_yearly,
            truth_yearly,
            bias_corrected_2023test,
            AIFS_inputs_2023_selection,
            pred_reduce="median",
            **common_kwargs,
        )
    )
    figs_nov_member.append(
        Viz_AllLeadsOverview(
            pred_yearly,
            truth_yearly,
            bias_corrected_2023test,
            AIFS_inputs_2023_selection,
            pred_reduce="member",
            member_index=0,
            **common_kwargs,
        )
    )

for i, fig_i in enumerate(figs_nov):
    fig_i.savefig(
        f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/Event_Nov_AllLeadsMedian_{i}.png",
        dpi=300,
        bbox_inches="tight",
    )
for i, fig_i in enumerate(figs_nov_member):
    fig_i.savefig(
        f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/Event_Nov_AllLeadsMember0_{i}.png",
        dpi=300,
        bbox_inches="tight",
    )

## 5)  Evaluate downscaling models — extract metrics

Prepare arguments for the metrics extraction

In [ ]:
# Shared arguments for all metric computations
METRIC_KWARGS = dict(
    point="mean",
    threshold=0.1,
    wet_threshold=0.1,
    heavy_threshold=1.0,
    fss_neighborhoods=[1, 5, 17],
    compute_spectral=True,
    compute_rank_hist=True,
)

# Common spatial window (same grid for all models)
WINDOW = bias_corrected_2.precip6h_cumulative_capped[0, 1]

Compute metrics for each considered downscaling model

In [ ]:
# MultiFT with new inference script
ds_metrics21 = compute_metrics_on_latlon_window_extended(
    pred=pred21,
    truth=truth21,
    window_latlon=WINDOW,
    **METRIC_KWARGS,
)
ds_metrics21.to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/metrics/metrics_mean_extra_clean_newInfScript.nc"
)

In [ ]:
# Yearly test: Nov–Dec 2023
ds_metrics_yearlyTest = compute_metrics_on_latlon_window_extended(
    pred=pred_yearly,
    truth=truth_yearly,
    window_latlon=bias_corrected_2023test.precip6h_cumulative_capped[0, 1],
    **METRIC_KWARGS,
)
ds_metrics_yearlyTest.to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/metrics/metrics_mean_extra_clean.nc"
)

In [ ]:
# New-hours test
ds_metrics_newHours = compute_metrics_on_latlon_window_extended(
    pred=pred_newHours,
    truth=truth_newHours,
    window_latlon=WINDOW,
    **METRIC_KWARGS,
)
ds_metrics_newHours.to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/metrics/metrics_mean_extra_clean.nc"
)

In [ ]:
# Yearly test across 2023
ds_metrics_2023ValStride5 = compute_metrics_on_latlon_window_extended(
    pred=pred_2023ValStride5,
    truth=truth_2023ValStride5,
    window_latlon=WINDOW,
    **METRIC_KWARGS,
)
ds_metrics_2023ValStride5.to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/metrics/metrics_mean_extra_clean.nc"
)

#### Extra analysis for Supplementary material

Seasonal evaluation

In [ ]:
MONTH_TO_SEASON = {
    12: "DJF",
    1: "DJF",
    2: "DJF",
    3: "MAM",
    4: "MAM",
    5: "MAM",
    6: "JJA",
    7: "JJA",
    8: "JJA",
    9: "SON",
    10: "SON",
    11: "SON",
}


def separate_by_season(da):
    """Split a DataArray into four seasonal subsets (DJF, MAM, JJA, SON)."""
    ds = da.assign_coords(
        season=("time", pd.DatetimeIndex(da.time.values).month.map(MONTH_TO_SEASON))
    )
    out = {}
    for name in ["DJF", "MAM", "JJA", "SON"]:
        subset = ds.sel(time=ds.season == name)
        print(f"{name}: {subset.sizes['time']} time steps")
        out[name] = subset
    return out["DJF"], out["MAM"], out["JJA"], out["SON"]


pred_seasons = list(separate_by_season(pred_2023ValStride5))
truth_seasons = list(separate_by_season(truth_2023ValStride5))

In [ ]:
season_names = ["DJF", "MAM", "JJA", "SON"]
for idx, name in enumerate(season_names):
    ds = compute_metrics_on_latlon_window_extended(
        pred=pred_seasons[idx],
        truth=truth_seasons[idx],
        window_latlon=WINDOW,
        **METRIC_KWARGS,
    )
    ds.to_netcdf(
        f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/metrics/metrics_mean_extra_clean_{idx + 1}.nc"
    )

Distribution-shift check: weekly model evaluated on 2023 only

In [ ]:
pred1_2023 = pred1.sel(time=pred1.time[-23:])
truth1_2023 = truth1.sel(time=truth1.time[-23:])

ds_metrics1_2023 = compute_metrics_on_latlon_window_extended(
    pred=pred1_2023,
    truth=truth1_2023,
    window_latlon=WINDOW,
    **METRIC_KWARGS,
)
ds_metrics1_2023.to_netcdf(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4/metrics/metrics_mean_extra_clean_2023.nc"
)

Autocorrelation check: 06 h and 18 h separately

In [ ]:
mask_18 = [t.endswith("T18:00:00") for t in pred_newHours.time.values]
mask_06 = [t.endswith("T06:00:00") for t in pred_newHours.time.values]

for mask, label in [(mask_18, "only18"), (mask_06, "only06")]:
    ds = compute_metrics_on_latlon_window_extended(
        pred=pred_newHours.sel(time=mask),
        truth=truth_newHours.sel(time=mask),
        window_latlon=WINDOW,
        **METRIC_KWARGS,
    )
    ds.to_netcdf(
        f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/metrics/metrics_mean_extra_clean_{label}.nc"
    )

#### Additional evaluation: PIT (Probability Integral Transform) analysis

In [ ]:
# Prepare PIT results for all considered models
pit_results = {}
for name, pred_i, truth_i in [
    ("UNet_λ2", pred1, truth1),
    ("MultiFT", pred2, truth2),
    ("AIFS", pred3, truth3),
    ("B4", pred4, truth4),
    ("Single_6h", pred5, truth5),
    ("Single_3d", pred6, truth6),
    ("Single_6d", pred7, truth7),
]:
    pit_results[name] = compute_pit_on_latlon_window(
        pred=pred_i,
        truth=truth_i,
        window_latlon=WINDOW,
        n_bins=20,
    )

# Compute results and visualize them
fig = plot_pit_comparison_combined(
    {k: pit_results[k] for k in ["UNet_λ2", "MultiFT", "AIFS", "B4"]},
)
fig.savefig(
    f"{RESULTS_ROOT}/PIT_multi_comparison_final2.png", dpi=300, bbox_inches="tight"
)

#### Spectral analysis: effective resolution

Run separately via `scripts/spectral_evaluation.py`.

## 6) Produce metrics tables / figures

Re-import metrics

In [ ]:
df_lam2 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)
df_MultiFT = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_MultiStepFineTuning2_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)
df_aifs = xr.open_dataset(
    f"{GENERATIONS_ROOT}/AIFSOrigPreds_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)
df_B4 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/BaselineB4_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)
df_single6h = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_6h_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)
df_single3days = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_3days_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)
df_single6days = xr.open_dataset(
    f"{GENERATIONS_ROOT}/SingleLead_6days_ValSampleStride4/metrics/metrics_mean_extra_clean.nc"
)

df_lam2_yearlyTests = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_YearlySplit2/metrics/metrics_mean_extra_clean.nc"
)
df_lam2_newHours = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/metrics/metrics_mean_extra_clean.nc"
)
ds_metrics_2023ValStride5 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/metrics/metrics_mean_extra_clean.nc"
)

ds_metrics_2023ValStride5_seasons = [
    xr.open_dataset(
        f"{GENERATIONS_ROOT}/UNet_lam2_2023ValStride5_YearlySplitTest/metrics/metrics_mean_extra_clean_{i}.nc"
    )
    for i in range(1, 5)
]

ds_metrics1_2023 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4/metrics/metrics_mean_extra_clean_2023.nc"
)
ds_metrics_newHours_18 = xr.open_dataset(
    f"{GENERATIONS_ROOT}/UNet_lam2_ValSampleStride4_6h_18hTest/metrics/metrics_mean_extra_clean_only18.nc"
)

Generate LaTeX tables

In [ ]:
model_results = {
    "UNet, No Curri, λ=2": df_lam2,
    "UNet Multi FineTuned": df_MultiFT,
    "B0: AIFS": df_aifs,
    "B4: Pixel LR, MultiVar": df_B4,
    "UNet Single 6h": df_single6h,
    "UNet Single 3day": df_single3days,
    "UNet Single 6day": df_single6days,
}

model_results_yearly = {
    "UNet$_{\\lambda=2}$, weekly": df_lam2,
    "UNet$_{\\lambda=2}$, weekly, new Hours": df_lam2_newHours,
    "UNet$_{\\lambda=2}$, weekly new Hours, only 18": ds_metrics_newHours_18,
    "UNet$_{\\lambda=2}$, yearly, on Nov-dec": df_lam2_yearlyTests,
    "UNet$_{\\lambda=2}$, yearly, val 2023": ds_metrics_2023ValStride5,
    "UNet$_{\\lambda=2}$, weekly, on 2023": ds_metrics1_2023,
    "UNet$_{\\lambda=2}$, yearly, val 2023, winter": ds_metrics_2023ValStride5_seasons[
        0
    ],
    "UNet$_{\\lambda=2}$, yearly, val 2023, spring": ds_metrics_2023ValStride5_seasons[
        1
    ],
    "UNet$_{\\lambda=2}$, yearly, val 2023, summer": ds_metrics_2023ValStride5_seasons[
        2
    ],
    "UNet$_{\\lambda=2}$, yearly, val 2023, autumn": ds_metrics_2023ValStride5_seasons[
        3
    ],
}

In [ ]:
# general metrics, averaged across leads
latex, df = generate_ensemble_latex_table_compact(
    model_results,
    metrics=["crps", "mse", "mse_wet", "csi", "avfss_5", "eS", "eA", "eL"],
    caption="Super-resolution model comparison",
)

In [ ]:
# detailed metric for multiple leads
LEADS_TO_SHOW = [0, 1, 2, 3, 4, 5, 6, 7, 8]

for metric in ["crps", "csi", "mse_wet"]:
    latex, df = generate_ensemble_latex_table_per_lead(
        model_results,
        metric=metric,
        positions_to_show=LEADS_TO_SHOW,
        debug=True,
    )

Additional visualizations: Spider plots

In [ ]:
SPIDER_COMMON = dict(
    metrics=["crps", "mae_wet", "csi", "avfss_5"],
    higher_is_better=["csi", "avfss_5", "spectral_correlation"],
    metric_labels={
        "crps": "CRPS",
        "mae_wet": "MAE_wet",
        "csi": "CSI",
        "avfss_5": "avFSS₅",
    },
    fill_alpha=0.07,
    line_alpha=0.9,
    figsize=(5, 6),
    title=None,
    show_values=True,
    quadrant_colors=["white", "darkgrey", "white", "darkgrey"],
)

fig_lowLeads = spider_plot_unified3(
    datasets={
        "swAIther (UNet$_{\\lambda=2}$)": df_lam2,
        "swAIther (Multi FT)": df_MultiFT,
        "AIFS raw": df_aifs,
        "B4 baseline": df_B4,
        "UNet Single 6h": df_single6h,
    },
    leads=[0, 2, 3, 7],
    model_lead_map={"UNet Single 6h": [0]},
    **SPIDER_COMMON,
)

fig_highLeads = spider_plot_unified3(
    datasets={
        "swAIther (UNet$_{\\lambda=2}$)": df_lam2,
        "swAIther (Multi FT)": df_MultiFT,
        "AIFS raw": df_aifs,
        "B4 baseline": df_B4,
        "UNet Single 3d": df_single3days,
        "UNet Single 6d": df_single6days,
    },
    leads=[11, 15, 19, 23],
    model_lead_map={"UNet Single 3d": [11], "UNet Single 6d": [23]},
    **SPIDER_COMMON,
)

fig_lowLeads.savefig(
    f"{RESULTS_ROOT}/spider_plot_LowLeads3.pdf", dpi=400, bbox_inches="tight"
)
fig_highLeads.savefig(
    f"{RESULTS_ROOT}/spider_plot_HighLeads3.pdf", dpi=400, bbox_inches="tight"
)